# P4 Banking77 intent classification: Kaggle runner

Thin wrapper: all logic lives in `src/textclf`.

Settings: **Accelerator GPU T4 x1**, **Internet ON**, then *Save Version -> Save & Run All*.
The run can be resumed: rerunning skips (model, seed) rows that already finished.

Download `p4_results.zip` from the Output tab at the end. It holds the metrics, per-class results,
error analysis, latency numbers, the model card and the test predictions. Model weights are left out.

In [ ]:
!git clone --depth 1 https://github.com/JDevender007/Artificial_Intelligence.git
%cd Artificial_Intelligence/04-intent-classification
!pip install -q -r requirements-gpu.txt && pip install -q -e . --no-deps

In [ ]:
!mkdir -p /kaggle/working/outputs/banking77
!python -c "import transformers, tokenizers, torch, sklearn, numpy; print('transformers', transformers.__version__); print('tokenizers', tokenizers.__version__); print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('scikit-learn', sklearn.__version__); print('numpy', numpy.__version__)" | tee /kaggle/working/outputs/banking77/env.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | tee -a /kaggle/working/outputs/banking77/env.txt
!lscpu | grep 'Model name' | tee -a /kaggle/working/outputs/banking77/env.txt

## Run `configs/banking77.yaml`
Downloads Banking77, makes the split, fits every model x seed, measures latency, then writes the error analysis and model card.

In [ ]:
!python scripts/run_experiment.py --config configs/banking77.yaml --set output_dir=/kaggle/working/outputs/banking77

In [ ]:
from pathlib import Path

for f in ['data_summary.md', 'results.md', 'latency.md', 'error_analysis.md']:
    p = Path('/kaggle/working/outputs/banking77') / f
    print(f'===== {f}')
    print(p.read_text() if p.exists() else '(missing)')

## Package results
The zip holds text results and predictions only (no weights).

In [ ]:
%cd /kaggle/working
!zip -qr p4_results.zip outputs -x '*/models/*'
!rm -rf outputs Artificial_Intelligence && ls -la /kaggle/working